Ce code est à utiliser avant les analyses de modèle : ya le traitement des variables issues de py AnalyseDesCUnivariee et traitement valeurs manquantes (tout est expliqué sur le latex)

Je met juste les lignes de code simple à prendre

In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("Base_ER_FR.xlsx")
df.columns

Index(['ID', 'AGE_CLI', 'B_MAT', 'B_RESMAT', 'CLASSACT', 'CSP', 'DARRET',
       'E_EAD', 'E_OFF', 'E_ONB', 'MREVAU', 'MREVNU', 'MREVTOT', 'MTECH',
       'NBIMP', 'NB_ECH', 'produit', 'RA', 'Tx', 'Unnamed: 19'],
      dtype='object')

code à copier :

In [3]:
#ANALYSE DES STATS DES
#B_MAT
df = df[(df["B_MAT"] <= 170) | (df["B_MAT"].isna())]

#drop variables dégénérées ou inutiles
cols_to_drop = ["Unnamed: 19", "MREVAU", "MREVNU"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

#DARRET
df["DARRET"] = pd.to_datetime(df["DARRET"].astype(str) + "01", format="%Y%m%d", errors="coerce")

if "E_EAD" in df.columns:
    df["EAD"] = df["E_EAD"]
    df.drop(columns=["E_EAD", "E_ONB"], inplace=True, errors="ignore")

df = df.dropna(subset=["EAD"])
# éventuellement ensuite : df = df[df["EAD"] >= 0]
df["RA"] = df["RA"].fillna(0)

for c in ["CLASSACT", "CSP", "produit"]:
    if c in df.columns:
        df[c] = df[c].astype("category")


#VALEURS MANQUANTES
df["Tx"] = df.groupby("produit", observed=True)["Tx"].transform(lambda x: x.fillna(x.median()))
df["Tx"] = df["Tx"].fillna(df["Tx"].median())
df["CSP"] = df["CSP"].cat.add_categories("Inconnu")
df["CSP"] = df["CSP"].fillna("Inconnu")
df["AGE_CLI"] = df["AGE_CLI"].fillna(df["AGE_CLI"].median())


# B_MAT/ B_RESMAT/ E_EAD
#df = df.dropna(subset=["B_MAT", "B_RESMAT", "E_EAD"])

# Revenus
df["MREVTOT"] = df["MREVTOT"].fillna(df["MREVTOT"].median())

#df["MREVTOT"] = df["MREVNU"] + df["MREVAU"]

#produit
df = df[df["produit"] != "AR"]

#RA
df = df[df["RA"] >= 0]

#Tx
df = df[df["Tx"] >= 0.5]

#ages 
df["AGE_PRET"] = df["B_MAT"] - df["B_RESMAT"]
df = df[(df["AGE_PRET"] >= 0) & (df["AGE_PRET"] <= df["B_MAT"])]
df["HORIZON_RES"] = df["B_RESMAT"].astype(int)


In [4]:
len(df)
print(((100000-len(df))/100000)*100)

8.921999999999999


pas contre faut bien penser a traiter CSP egalement avant les analyes

Proposition:

In [5]:
csp_map = {
    # SALARIÉS 
    70.0: "SALARIE",
    52.0: "SALARIE",
    46.0: "SALARIE",
    47.0: "SALARIE",
    48.0: "SALARIE",
    54.0: "SALARIE",
    55.0: "SALARIE",
    56.0: "SALARIE",
    36.0: "SALARIE",
    31.0: "SALARIE",
    32.0: "SALARIE",
    45.0: "SALARIE",
    53.0: "SALARIE",

    # INACTIFS
    60.0: "INACTIF",
    64.0: "INACTIF",

    # INDÉPENDANTS / AUTRES ACTIFS
    42.0: "INDEPENDANT",
    82.0: "INDEPENDANT",
    87.0: "INDEPENDANT",
    23.0: "INDEPENDANT"
}

df["CSP_grp"] = df["CSP"].map(csp_map)
df["CSP_grp"] = df["CSP_grp"].fillna("AUTRE")


csp_check = (
    df["CSP_grp"]
    .value_counts(dropna=False)
    .to_frame("effectifs")
)
csp_check["pourcentage"] = 100 * csp_check["effectifs"] / len(df)
csp_check



,effectifs,pourcentage
CSP_grp,,
SALARIE,72599,79.710797
INACTIF,10737,11.788796
AUTRE,4273,4.691583
INDEPENDANT,3469,3.808823


### Taux

### Dernieres verifications 

In [6]:
nb_lignes = len(df)
print(nb_lignes)

91078


In [7]:
base = pd.read_excel("Base_ER_FR.xlsx")


In [8]:
nb_lignes = len(base)
print(nb_lignes)

100000


## Etape 2 : Calcul CRD et ER (formule professeur)

### Formule utilisee

```
CRD = MTECH × B_RESMAT - RA
ER = RA / CRD (clippe entre 0 et 1)
```

### Capital initial pour agregation

Pour l'agregation par cohorte, on utilise :
```
Capital_initial = MTECH × B_MAT
```

Cela donne le capital initial approximatif de chaque pret.

In [9]:
# Calcul CRD et ER selon formule prof
df["CRD"] = df["MTECH"] * df["B_RESMAT"] - df["RA"]
df["ER_obs"] = df["RA"] / df["CRD"]
df["ER_obs"] = df["ER_obs"].clip(lower=0, upper=1)

# FLAG_ER
df["FLAG_ER"] = (df["RA"] > 0).astype(int)

# Filtrer CRD invalides
print(f"Contrats avec CRD <= 0 : {(df['CRD'] <= 0).sum()} ({(df['CRD'] <= 0).mean():.2%})")
df = df[df["CRD"] > 0]

# Capital initial pour agregation par cohorte
df["Capital_initial"] = df["MTECH"] * df["B_MAT"]

# Reconstituer la date d'octroi et la cohorte
df["Date_origination"] = df["DARRET"] - pd.to_timedelta(df["AGE_PRET"] * 30, unit='D')
df["Cohorte"] = df["Date_origination"].dt.to_period('Q')  # TRIMESTRE pour plus de stabilite

print(f"\nDonnees preparees : {len(df)} contrats")
print(f"Capital initial moyen : {df['Capital_initial'].mean():.2f} EUR")
print(f"CRD moyen : {df['CRD'].mean():.2f} EUR")
print(f"ER_obs moyen : {df['ER_obs'].mean():.4f}")
print(f"Nombre de cohortes : {df['Cohorte'].nunique()}")

Contrats avec CRD <= 0 : 27493 (30.19%)

Donnees preparees : 63585 contrats
Capital initial moyen : 19325.38 EUR
CRD moyen : 3210.51 EUR
ER_obs moyen : 0.9164
Nombre de cohortes : 69


In [10]:
df.to_csv("Export_ER_Agrege.csv", index=False, sep=';')

et là encore il reste une dégénérésence dans le sens ou j'ai bcp d'obs encore sur une catégorie et moins sur les autres  (revoir avec INSEE)